# Hugging Face Tutorial — Getting Started with `transformers`

**What Hugging Face actually is** — a few separate but connected pieces:
- **The Hub** (huggingface.co) — a website hosting hundreds of thousands of
  pretrained models, datasets, and demo apps ("Spaces"), each with a model card
  describing what it does and how it was trained.
- **`transformers`** — the Python library that downloads a model from the Hub
  and runs it (this is what we'll use here).
- **`datasets`** — a library for loading and processing datasets, often paired
  with `transformers` for fine-tuning (not covered in this tutorial).
- **`huggingface_hub`** — a lightweight library for searching/downloading from
  the Hub programmatically.

Run the setup cell below first.

In [2]:
import transformers
from transformers.utils import logging

print("transformers version:", transformers.__version__)
logging.disable_progress_bar()

transformers version: 5.16.1


---
## 1) The fastest way in: `pipeline()`

`pipeline()` is a high-level wrapper: give it a task name and (optionally) a
model name, and it downloads the tokenizer + model for you and handles all the
pre/post-processing. It's the right entry point for "I just want to try a task",
and it's exactly what powers the widgets on Hugging Face model pages.

In [3]:
from transformers import pipeline

# Sentiment analysis — no model name given, so it uses a sensible default
classifier = pipeline("sentiment-analysis")
print(5*"-")
print(classifier("I really enjoyed learning about transformers today!"))
print(classifier("This tutorial is way too confusing."))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


-----
[{'label': 'POSITIVE', 'score': 0.9997525811195374}]
[{'label': 'NEGATIVE', 'score': 0.9997068047523499}]


Some other common tasks — each `pipeline()` call downloads a different
default model suited to that task the first time you run it:

In [4]:
# Text generation (decoder-only, GPT-2-style)
generator = pipeline("text-generation", model="gpt2")
print(5*"-")
print(generator("Once upon a time,", max_new_tokens=20, num_return_sequences=1))

print(40*"=")
# Fill-mask (encoder-only, BERT-style)
unmasker = pipeline("fill-mask", model="bert-base-uncased")
print(5*"-")
print(unmasker("Paris is the [MASK] of France.", top_k=3))
print(40*"=")
# Zero-shot classification — classify text into labels it was never trained on
print(5*"-")
zero_shot = pipeline("zero-shot-classification")
print(zero_shot(
    "This new phone has an amazing camera and battery life.",
    candidate_labels=["science","technology", "sports", "politics"],
))

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


-----


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'Once upon a time, my father was a child of the royal family. He was a great man who died a great human'}]


[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


-----
[{'score': 0.9969332218170166, 'token': 3007, 'token_str': 'capital', 'sequence': 'paris is the capital of france.'}, {'score': 0.0005914872162975371, 'token': 2540, 'token_str': 'heart', 'sequence': 'paris is the heart of france.'}, {'score': 0.0004378740268293768, 'token': 2415, 'token_str': 'center', 'sequence': 'paris is the center of france.'}]
-----
{'sequence': 'This new phone has an amazing camera and battery life.', 'labels': ['technology', 'sports', 'science', 'politics'], 'scores': [0.9839093685150146, 0.009241260588169098, 0.005076330620795488, 0.0017730933614075184]}


---
## 2) What `pipeline()` is doing under the hood

`pipeline()` is convenient, but it's worth seeing the two pieces it's hiding:
a **tokenizer** (text → token IDs) and a **model** (token IDs → logits). This
is the same `AutoTokenizer` / `AutoModel` pattern you've already used in the
playground notebook — `pipeline()` just wraps it for you.

In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Model name
model_name = "Falconsai/text_summarization"

# 2. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. Load model
model = AutoModelForSeq2SeqLM.from_pretrained(model_name,)

# 4. Input text
text = """
The tower is 324 metres (1,063 ft) tall, about the same height as an 81-storey
building, and the tallest structure in Paris. Its base is square, measuring
125 metres (410 ft) on each side. During its construction, the Eiffel Tower
surpassed the Washington Monument to become the tallest man-made structure
in the world, a title it held for 41 years until the Chrysler Building in
New York City was finished in 1930. It was the first structure to reach a
height of 300 metres.
"""

# 5. Tokenize
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True
)

# 6. Generate summary
outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

# 7. Convert token IDs back to text
summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Summary:")
print(summary)

Summary:
, and the tallest structure in Paris. It is 324 metres (1,063 ft) tall, about the same height as an 81-storey building, and the tallest structure in Paris. Its base is


In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

text = "I really enjoyed learning about transformers today!"

# Step 1: tokenizer turns text into IDs the model understands
inputs = tokenizer(text, return_tensors="pt")
print("input_ids:", inputs["input_ids"])
print("attention_mask:", inputs["attention_mask"])
print("tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

# Step 2: model turns IDs into raw logits
with torch.no_grad():
    logits = model(**inputs).logits
print("raw logits:", logits)

# Step 3: softmax turns logits into probabilities, argmax picks the label
probs = torch.softmax(logits, dim=-1)
predicted_class = torch.argmax(probs, dim=-1).item()
print("probabilities:", probs)
print("predicted label:", model.config.id2label[predicted_class])

input_ids: tensor([[  101,  1045,  2428,  5632,  4083,  2055, 19081,  2651,   999,   102]])
attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
tokens: ['[CLS]', 'i', 'really', 'enjoyed', 'learning', 'about', 'transformers', 'today', '!', '[SEP]']
raw logits: tensor([[-4.0287,  4.2756]])
probabilities: tensor([[2.4738e-04, 9.9975e-01]])
predicted label: POSITIVE


This is exactly the `softmax(logits)` step from the attention fundamentals —
here it's applied once at the very end, over class labels, rather than inside
attention over tokens.

---
## 3) Batching multiple sentences: padding, truncation, attention_mask

Real usage almost always means processing more than one sentence at once —
which is exactly where padding masks (fundamentals §3.6) become necessary.

In [13]:
sentences = [
    "I loved this movie.",
    "This was the worst experience I've ever had, and I want a refund.",
    "It was okay, nothing special.",
]

batch = tokenizer(
    sentences,
    padding=True,       # pad shorter sequences up to the longest one in the batch
    truncation=True,    # cut off anything longer than the model's max length
    return_tensors="pt",
)

print("input_ids shape:", batch["input_ids"].shape)
print("attention_mask:\n", batch["attention_mask"])

with torch.no_grad():
    logits = model(**batch).logits
predictions = torch.argmax(logits, dim=-1)

for sentence, pred in zip(sentences, predictions):
    print(f"{model.config.id2label[pred.item()]:10s} -> {sentence}")

input_ids shape: torch.Size([3, 20])
attention_mask:
 tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
POSITIVE   -> I loved this movie.
NEGATIVE   -> This was the worst experience I've ever had, and I want a refund.
NEGATIVE   -> It was okay, nothing special.


Notice the `attention_mask` — it's `1` for real tokens and `0` for padding, so
the model knows to ignore the padding positions, exactly like the padding-mask
mechanism described in the fundamentals notes.

---
## 4) Saving and loading a model locally

Once you've downloaded a model, you don't need to re-download it every time —
`save_pretrained()` / `from_pretrained()` work the same way for local folders
as they do for Hub model names.

In [14]:
save_dir = "./my_local_model"

tokenizer.save_pretrained(save_dir)
model.save_pretrained(save_dir)

# Reload from disk instead of the Hub
local_tokenizer = AutoTokenizer.from_pretrained(save_dir)
local_model = AutoModelForSequenceClassification.from_pretrained(save_dir)

print("Reloaded model config:", local_model.config.model_type)

Reloaded model config: distilbert
